# Apply an approved agent instruction

The narrow half of the remediation loop. This notebook is **generated**;
edit `validation/build_agent_remediation_notebook.py` and regenerate.

## Why this is separate from agent_remediate

It installs `fabric-data-agent-sdk` at run time. The repo has already
lost a scheduled job to a `%pip install` pulling new builds of pydantic
and anyio over the ones the Spark runtime ships, so that risk is kept
away from the path that writes to the semantic model.

`agent_remediate` reaches this with `notebookutils.notebook.run()`, a
reference run in its own session, and only when there is agent-targeted
work to do.

## What an agent instruction can change

How an answer reads, and nothing else. Agent instructions are applied
after the query has run, so they cannot change a value, a filter or a
grouping. This notebook re-checks `instruction_target` rather than
trusting its caller, because a model-class fix applied here would be
recorded as persisted and change nothing.

## What it will not do

- Write without a named approver
- Replace instructions it could not first read
- Rewrite or delete text a human wrote. It appends under one heading
- Record anything as applied unless the write read back identically


## 1. Parameters

In [ ]:
WORKSPACE_ID = ""
DATA_AGENT_ID = ""
DATA_AGENT_NAME = ""  # the item's display name, which the SDK takes
KUSTO_URI = ""
KUSTO_DB = "EH_AgentEval"

# Which approvals to apply. agent_remediate passes these, comma separated.
APPROVAL_IDS = ""
APPROVED_BY = ""

# Coerced below rather than trusted. A parameter injected by a reference run
# arrives as the string "false", and a non-empty string is truthy in Python.
DRY_RUN = True


## 2. The SDK

The one place in this repo that installs anything at run time. It is
here rather than in `agent_remediate` because a dependency that breaks
the Spark runtime must not be able to break the path that writes to the
semantic model, and this notebook is reached by a reference run rather
than `%run`, so it gets its own session.

In [ ]:
%pip install -q -U fabric-data-agent-sdk


## 3. Find the approved work

By approval id, passed in by the caller, and re-filtered here so a
stale or already applied id cannot be applied twice.

In [ ]:
import json
import urllib.request
import uuid
from datetime import datetime, timezone

import notebookutils

DRY_RUN = str(DRY_RUN).strip().lower() not in ("false", "0", "no", "")
print(f"DRY_RUN resolved to {DRY_RUN}")

if not APPROVED_BY.strip():
    raise ValueError(
        "APPROVED_BY is empty. A governed change records who approved it."
    )

TARGET_DATA_AGENT = "data_agent"

kusto_token = notebookutils.credentials.getToken(KUSTO_URI)


def kusto(csl, endpoint="query"):
    request = urllib.request.Request(
        f"{KUSTO_URI}/v1/rest/{endpoint}",
        data=json.dumps({"db": KUSTO_DB, "csl": csl}).encode("utf-8"),
        method="POST",
        headers={"Authorization": f"Bearer {kusto_token}",
                 "Content-Type": "application/json"},
    )
    with urllib.request.urlopen(request, timeout=120) as response:
        return json.loads(response.read().decode("utf-8"))


def rows(result):
    table = result["Tables"][0]
    names = [c["ColumnName"] for c in table["Columns"]]
    return [dict(zip(names, row)) for row in table["Rows"]]


wanted = [a.strip() for a in APPROVAL_IDS.split(",") if a.strip()]
if not wanted:
    print("no approval ids passed, nothing to do")
    pending = []
else:
    quoted = ", ".join(f'"{a}"' for a in wanted)
    pending = rows(kusto(f"""
        eval_approvals
        | where approval_id in ({quoted})
        | where decision == "approved"
        | join kind=leftanti (
            eval_remediations
            | where persisted == true
            | distinct approval_id
          ) on approval_id
    """))

# Re-checked rather than trusted. The caller already filtered by target, but
# this notebook writes to a governed item and a caller that got it wrong would
# otherwise apply a model-class fix somewhere it can never work.
misrouted = [r for r in pending if r["instruction_target"] != TARGET_DATA_AGENT]
if misrouted:
    raise ValueError(
        f"{len(misrouted)} approval(s) are not agent targeted: "
        + ", ".join(r["question_id"] for r in misrouted)
        + ". Agent instructions are applied after the query has run and cannot "
        "change a value, so applying these here would look like a change and "
        "do nothing."
    )

print(f"{len(pending)} agent-targeted approval(s) to apply")
for row in pending:
    print(f"  {row['question_id']} by {row['approved_by']}")
    print(f"      {row['proposed_instruction'][:160]}")


## 4. Merge and apply

`update_settings` replaces the whole instruction value, so the current
text is read first and appended to. A run that cannot read it refuses.

In [ ]:
from fabric.dataagent.client import FabricDataAgentManagement

AGENT_HEADING = "## Automated remediation"


def merge_instruction(existing, instruction):
    """Append one line under the loop's heading, idempotently.

    The same shape as the model path: never rewrite or delete text a human
    wrote, and adding a line that is already there is a no-op rather than a
    duplicate.
    """
    existing = existing or ""
    if instruction.strip() and instruction.strip() in existing:
        return existing, False
    if AGENT_HEADING in existing:
        return existing.rstrip() + "\n" + instruction + "\n", True
    separator = "\n\n" if existing.strip() else ""
    return (
        existing.rstrip() + separator + AGENT_HEADING + "\n\n"
        + "Added by the evaluation loop after a human approved each line.\n\n"
        + instruction + "\n"
    ), True


applied = []
if pending:
    agent = FabricDataAgentManagement(DATA_AGENT_NAME or DATA_AGENT_ID)

    # Read before write. If the current instructions cannot be established,
    # this refuses rather than replacing them: update_settings sets the whole
    # value, so a wrong read here would silently delete whatever a person had
    # written by hand.
    configuration = agent.get_configuration()
    current = getattr(configuration, "instructions", None)
    if current is None:
        current = getattr(configuration, "ai_instructions", None)
    if current is None and isinstance(getattr(configuration, "value", None), dict):
        current = configuration.value.get("instructions")
    if current is None:
        raise ValueError(
            "Could not read the agent's current instructions, so this run will "
            "not write. update_settings replaces the whole value, and writing "
            "without a reliable read would delete whatever a person wrote by "
            "hand. Inspect get_configuration() and update this cell."
        )

    proposed = current
    for row in pending:
        proposed, changed = merge_instruction(proposed, row["proposed_instruction"])
        if not changed:
            print(f"already present, nothing to add for {row['question_id']}")
        applied.append(row)

    print("--- current ---")
    print(current[-600:] if current else "(empty)")
    print("--- proposed ---")
    print(proposed[-600:])

    if DRY_RUN:
        print("\nDRY_RUN, nothing written")
    elif proposed == current:
        print("\nno change to write")
    else:
        agent.update_settings(ai_instructions=proposed)

        # Read back. execute-and-hope is not evidence, and the identity this
        # runs as may not have write access to the agent, which produces a
        # silent no-op rather than an error.
        after = agent.get_configuration()
        landed = getattr(after, "instructions", None) or getattr(
            after, "ai_instructions", None
        )
        if landed != proposed:
            raise RuntimeError(
                "the write did not land. The agent instructions read back "
                "differently from what was sent, so nothing is recorded as "
                "applied."
            )
        print("\nwrite verified")


## 5. Record what happened

Into `eval_remediations`, the same table the model path writes, so the
loop has one history. The mirror pipeline carries it back to SQL for
the report.

In [ ]:
# Recorded to the same table as the model path, so the dashboard and the
# report show one history rather than two. verified stays false until an
# evaluation run proves the question actually improved.
try:
    executing_identity = notebookutils.runtime.context.get("userName", "unknown")
except Exception:  # noqa: BLE001
    executing_identity = "unknown"

now = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S.%fZ")


def escape(value):
    return (value or "").replace("\\", "\\\\").replace('"', '\\"')


written = 0
for row in applied:
    if DRY_RUN:
        print(f"DRY_RUN, not recording {row['question_id']}")
        continue
    kusto(
        ".set-or-append eval_remediations <| print "
        f'remediation_id="{uuid.uuid4()}", '
        f"recorded_ts=datetime({now}), "
        f"applied_ts=datetime({now}), "
        f'approval_id="{escape(row["approval_id"])}", '
        f'question_id="{escape(row["question_id"])}", '
        f'instruction_target="{escape(row["instruction_target"])}", '
        f'instruction="{escape(row["proposed_instruction"])}", '
        f'approved_by="{escape(row["approved_by"])}", '
        f'applied_by="{escape(executing_identity)}", '
        "dry_run=false, persisted=true, verified=false, "
        "verified_ts=datetime(null), verified_run_id=\"\"",
        endpoint="mgmt",
    )
    written += 1

print(f"recorded {written} remediation(s)")
print("verified stays false until an evaluation run proves the fix worked")
